In [ ]:
# In[1]:

import pandas as pd

# Load metric.csv
df_metric = pd.read_csv('metric.csv')
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Aggregate metric.csv by (cmdb_id, kpi_name)
metric_grp = df_metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)
metric_agg = metric_grp.agg(
    count=('value', 'count'),
    min_ts=('timestamp', 'min'),
    max_ts=('timestamp', 'max'),
    p95=('value', lambda s: s.quantile(0.95))
).reset_index()

# Format timestamps as ISO UTC strings and round p95
metric_agg['min_ts'] = metric_agg['min_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
metric_agg['max_ts'] = metric_agg['max_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
metric_agg['p95'] = metric_agg['p95'].round(6)

# Keep top 50 by count desc
metric_agg = metric_agg.sort_values('count', ascending=False).head(50)

# Load trace.csv
df_trace = pd.read_csv('trace.csv')
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Aggregate trace.csv by (cmdb_id, trace_name)
trace_grp = df_trace.groupby(['cmdb_id', 'trace_name'], dropna=False)
trace_agg = trace_grp.agg(
    count=('value', 'count'),
    min_ts=('timestamp', 'min'),
    max_ts=('timestamp', 'max'),
    p95=('value', lambda s: s.quantile(0.95))
).reset_index()

# Format timestamps and round p95
trace_agg['min_ts'] = trace_agg['min_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
trace_agg['max_ts'] = trace_agg['max_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
trace_agg['p95'] = trace_agg['p95'].round(6)

# Keep top 50 by count desc
trace_agg = trace_agg.sort_values('count', ascending=False).head(50)

# Load log.csv
df_log = pd.read_csv('log.csv')
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Aggregate log.csv by (cmdb_id, log_name)
log_grp = df_log.groupby(['cmdb_id', 'log_name'], dropna=False)
log_agg = log_grp.agg(
    count=('value', 'count'),
    min_ts=('timestamp', 'min'),
    max_ts=('timestamp', 'max'),
    p95=('value', lambda s: s.quantile(0.95))
).reset_index()

# Format timestamps and round p95
log_agg['min_ts'] = log_agg['min_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
log_agg['max_ts'] = log_agg['max_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
log_agg['p95'] = log_agg['p95'].round(6)

# Keep top 50 by count desc
log_agg = log_agg.sort_values('count', ascending=False).head(50)

# Load error_logs.csv
df_error = pd.read_csv('error_logs.csv')
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# Aggregate error logs by cmdb_id: count, min_ts, max_ts
error_summary = df_error.groupby('cmdb_id', dropna=False).agg(
    count=('message', 'size'),
    min_ts=('timestamp', 'min'),
    max_ts=('timestamp', 'max')
).reset_index()

# Sample up to 5 messages per cmdb_id
sample_msgs = df_error.groupby('cmdb_id')['message'].apply(lambda s: s.dropna().astype(str).head(5).tolist()).reset_index()
error_agg = error_summary.merge(sample_msgs, on='cmdb_id', how='left')

# Format timestamps
error_agg['min_ts'] = error_agg['min_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
error_agg['max_ts'] = error_agg['max_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Limit to top 20 cmdb_id by count desc
error_agg = error_agg.sort_values('count', ascending=False).head(20)

# Display the aggregated results (compact)
metric_agg, trace_agg, log_agg, error_agg

```
Out[1]:
```
```python
# Summarize the aggregated results in plain English as a single compact string.
# Reuse existing aggregated DataFrames: metric_agg, trace_agg, log_agg, error_agg

def fmt_rows(df, n=3, cols=None):
    rows = []
    if df is None or df.shape[0] == 0:
        return "none"
    for _, r in df.head(n).iterrows():
        parts = []
        if cols is None:
            parts = [f"{c}={r[c]}" for c in df.columns]
        else:
            for c in cols:
                parts.append(f"{c}={r[c]}")
        rows.append("[" + ", ".join(parts) + "]")
    return "; ".join(rows)

# Prepare small summaries for top 3 groups in each aggregated table
metric_top3 = fmt_rows(metric_agg, n=3, cols=['cmdb_id', 'kpi_name', 'count', 'p95', 'min_ts', 'max_ts'])
trace_top3 = fmt_rows(trace_agg, n=3, cols=['cmdb_id', 'trace_name', 'count', 'p95', 'min_ts', 'max_ts'])
log_top3 = fmt_rows(log_agg, n=3, cols=['cmdb_id', 'log_name', 'count', 'p95', 'min_ts', 'max_ts'])

# error_agg may be empty
if 'error_agg' in globals() and isinstance(error_agg, type(metric_agg)) and error_agg.shape[0] > 0:
    error_summary = "Top error groups: " + fmt_rows(error_agg, n=5, cols=['cmdb_id', 'count', 'min_ts', 'max_ts', 'message'])
else:
    error_summary = "No error log entries found (error_logs.csv aggregated result is empty)."

summary = (
    "Summary of telemetry aggregates:\n"
    f"1) metric.csv (top 3 groups by count): {metric_top3}\n"
    f"2) trace.csv (top 3 groups by count): {trace_top3}\n"
    f"3) log.csv (top 3 groups by count): {log_top3}\n"
    f"4) error_logs.csv: {error_summary}\n\n"
    "Notes: timestamps are shown as ISO UTC strings; p95 values are included per group. "
    "Use the above grouped lines to inspect the highest-count KPI/trace/log groups and their p95s."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  count                min_ts                max_ts           p95
0               adservice         cpu     25  2024-01-20T04:39:00Z  2024-01-20T05:03:00Z  2.500910e+00
1               adservice      diskio     25  2024-01-20T04:39:00Z  2024-01-20T05:03:00Z  9.038738e+04
2               adservice  latency-50     25  2024-01-20T04:39:00Z  2024-01-20T05:03:00Z  3.084000e-03
3               adservice  latency-90     25  2024-01-20T04:39:00Z  2024-01-20T05:03:00Z  4.752000e-03
4               adservice         mem     25  2024-01-20T04:39:00Z  2024-01-20T05:03:00Z  1.051432e+08
..                    ...         ...    ...                   ...                   ...           ...
46         paymentservice  latency-90     25  2024-01-20T04:39:00Z  2024-01-20T05:03:00Z  9.073000e-03
48         paymentservice      socket     25  2024-01-20T04:39:00Z  2024-01-20T05:03:00Z  3.000000e+00
47         paymentservice         mem     25  2024-01-20T04:39:00Z  2024-01-20T05:03:00Z  3.361440e+07
49         paymentservice    workload     25  2024-01-20T04:39:00Z  2024-01-20T05:03:00Z  1.224397e+00
50  productcatalogservice         cpu     25  2024-01-20T04:39:00Z  2024-01-20T05:03:00Z  4.231176e+00

[50 rows x 6 columns],             cmdb_id                               trace_name  count                min_ts                max_ts          p95
67  frontendservice       trace.to_currencyservice.row_count     25  2024-01-20T04:39:00Z  2024-01-20T05:03:00Z  2439.400000
66  frontendservice      trace.to_currencyservice.error_rate     25  2024-01-20T04:39:00Z  2024-01-20T05:03:00Z     0.000000
65  frontendservice    trace.to_currencyservice.duration_p95     25  2024-01-20T04:39:00Z  2024-01-20T05:03:00Z     0.000262
64  frontendservice   trace.to_currencyservice.duration_mean     25  2024-01-20T04:39:00Z  2024-01-20T05:03:00Z     0.000690
39  currencyservice     trace.from_frontendservice.row_count     25  2024-01-20T04:39:00Z  2024-01-20T05:03:00Z  2439.400000
..              ...                                      ...    ...                   ...                   ...          ...
45     emailservice  trace.from_checkoutservice.duration_p95     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z     0.074092
46     emailservice    trace.from_checkoutservice.error_rate     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z     0.000000
47     emailservice     trace.from_checkoutservice.row_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z    32.000000
48     emailservice            trace.from_root.duration_mean     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z     0.000055
49     emailservice             trace.from_root.duration_p95     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z     0.000108

[50 rows x 6 columns],                   cmdb_id         log_name  count                min_ts                max_ts      p95
0               adservice  log.error_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z     0.00
1               adservice    log.row_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z   475.70
2             cartservice  log.error_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z     0.00
3             cartservice    log.row_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z   821.75
4         checkoutservice  log.error_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z     0.00
5         checkoutservice    log.row_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z    96.00
6         currencyservice  log.error_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z     0.00
7         currencyservice    log.row_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z  2545.80
8            emailservice  log.error_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z     0.00
9            emailservice    log.row_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z    32.00
10               frontend  log.error_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z     0.00
11               frontend    log.row_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z  2809.70
12         paymentservice  log.error_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z     0.00
13         paymentservice    log.row_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z    64.00
14  recommendationservice  log.error_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z     0.00
15  recommendationservice    log.row_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z   599.70
19        shippingservice    log.row_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z   501.40
18        shippingservice  log.error_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z     0.00
17                  redis    log.row_count      5  2024-01-20T04:41:00Z  2024-01-20T05:01:00Z     5.00
16                  redis  log.error_count      5  2024-01-20T04:41:00Z  2024-01-20T05:01:00Z     0.00, Empty DataFrame
Columns: [cmdb_id, count, min_ts, max_ts, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse df_metric if available; otherwise load and parse timestamps
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed correctly (idempotent)
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start = pd.to_datetime('2024-01-20T04:36:00Z', utc=True)
end = pd.to_datetime('2024-01-20T05:06:00Z', utc=True)

# Compute global P95 per (cmdb_id, kpi_name) from the full metric series (global threshold)
p95_global = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)

# Filter metric rows for specified services and the incident window
services = ["checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"]
df_window = df_metric[
    df_metric['cmdb_id'].isin(services) &
    (df_metric['timestamp'] >= start) &
    (df_metric['timestamp'] <= end)
].copy()

# If no in-window rows, produce empty summary with requested columns
if df_window.shape[0] == 0:
    summary_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','count','anomaly_count','earliest_anom_ts','max_value','p95','ratio'
    ])
else:
    # Merge global p95 into the windowed rows
    df_win_with_p95 = df_window.merge(p95_global, on=['cmdb_id','kpi_name'], how='left')

    # Determine per-row anomaly: value >= global p95 and p95 not NaN
    df_win_with_p95['is_anom'] = (df_win_with_p95['p95'].notna()) & (df_win_with_p95['value'] >= df_win_with_p95['p95'])

    # Aggregate within the window
    grouped = df_win_with_p95.groupby(['cmdb_id','kpi_name'], dropna=False)
    summary = grouped.agg(
        count = ('value', 'count'),
        anomaly_count = ('is_anom', 'sum'),
        max_value = ('value', 'max'),
        p95 = ('p95', 'first')
    ).reset_index()

    # earliest anomaly timestamp per group (if any)
    earliest = (
        df_win_with_p95[df_win_with_p95['is_anom']]
        .groupby(['cmdb_id','kpi_name'], dropna=False)['timestamp']
        .min()
        .reset_index(name='earliest_anom_ts')
    )

    summary = summary.merge(earliest, on=['cmdb_id','kpi_name'], how='left')

    # Format earliest_anom_ts as ISO UTC strings; leave as NA when no anomaly
    summary['earliest_anom_ts'] = pd.to_datetime(summary['earliest_anom_ts'], utc=True)
    summary['earliest_anom_ts'] = summary['earliest_anom_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    summary['earliest_anom_ts'] = summary['earliest_anom_ts'].where(summary['anomaly_count']>0, pd.NA)

    # Compute ratio = max_value / p95, handle p95==0 and p95 NaN
    def compute_ratio(row):
        p = row['p95']
        m = row['max_value']
        if pd.isna(p):
            return np.nan
        if p == 0:
            return np.inf
        return m / p

    summary['ratio'] = summary.apply(compute_ratio, axis=1)

    # Round numeric columns for compactness
    summary['p95'] = pd.to_numeric(summary['p95'], errors='coerce').round(6)
    summary['max_value'] = pd.to_numeric(summary['max_value'], errors='coerce').round(6)
    summary['ratio'] = pd.to_numeric(summary['ratio'], errors='coerce')

    # Select and order columns, sort by anomaly_count desc then ratio desc, limit to top 20
    summary_df = summary[[
        'cmdb_id','kpi_name','count','anomaly_count','earliest_anom_ts','max_value','p95','ratio'
    ]].sort_values(['anomaly_count','ratio'], ascending=[False, False]).head(20).reset_index(drop=True)

# Display the compact summary table
summary_df

```
Out[2]:
```
Summary of findings (plain English):

- The highest-count anomalies (anomaly_count = 25) are for several "diskio" groups across currencyservice, emailservice, productcatalogservice and recommendationservice. These all have global p95 = 0.0 and max_value = 0.0, so every in-window point meets value >= p95; the computed ratio is infinite. Treat these with caution — a p95 of zero makes the "anomaly" flag and ratio not meaningful.

- Several groups also show anomaly_count = 25 for "socket" KPIs (checkoutservice, currencyservice, productcatalogservice, recommendationservice). For these p95 equals the observed max (ratio = 1.0), so values consistently equal the p95 across the window.

- The most notable non-trivial increases (p95 > 0 and ratio > 1) are:
  1. emailservice, latency-50 — anomaly_count = 25? (table shows 25 total points, anomalies = 2), earliest anomaly 2024-01-20T04:51:00Z, max=0.029478, p95=0.024519, ratio ≈ 1.202 (largest relative uplift).
  2. checkoutservice, cpu — anomaly_count = 2, earliest 2024-01-20T04:43:00Z, max=0.548793, p95=0.511585, ratio ≈ 1.073.
  3. currencyservice, latency-90 — anomaly_count = 2, earliest 2024-01-20T04:43:00Z, max=0.057071, p95=0.053438, ratio ≈ 1.068.
  4. checkoutservice, latency-50 — anomaly_count = 2, earliest 2024-01-20T04:52:00Z, max=0.163646, p95=0.154972, ratio ≈ 1.056.
  5. recommendationservice, cpu — anomaly_count = 2, earliest 2024-01-20T04:49:00Z, max=3.054354, p95=2.907853, ratio ≈ 1.050.

- General notes:
  - Most groups have 25 points in-window (full coverage of the minute-bucketed window).
  - Diskio groups with p95 == 0 and socket groups with ratio == 1.0 should be interpreted cautiously — they may be artifacts of the global threshold or steady-state values, not clear-service degradations.
  - The strongest meaningful signal is emailservice latency-50 (ratio ≈ 1.20) with earliest in-window anomaly at 2024-01-20T04:51:00Z; several CPU/latency/workload KPIs show modest (~5–7%) exceedances of their global p95s.

If you want, I can return a short ranked table of the top non-trivial anomalies (p95 > 0) or help investigate the diskio/socket groups further.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count  anomaly_count      earliest_anom_ts   max_value         p95     ratio
0         currencyservice      diskio     25             25  2024-01-20T04:39:00Z    0.000000    0.000000       inf
1            emailservice      diskio     25             25  2024-01-20T04:39:00Z    0.000000    0.000000       inf
2   productcatalogservice      diskio     25             25  2024-01-20T04:39:00Z    0.000000    0.000000       inf
3   recommendationservice      diskio     25             25  2024-01-20T04:39:00Z    0.000000    0.000000       inf
4         checkoutservice      socket     25             25  2024-01-20T04:39:00Z    9.000000    9.000000  1.000000
5         currencyservice      socket     25             25  2024-01-20T04:39:00Z    3.000000    3.000000  1.000000
6   productcatalogservice      socket     25             25  2024-01-20T04:39:00Z    3.000000    3.000000  1.000000
7   recommendationservice      socket     25             25  2024-01-20T04:39:00Z    4.000000    4.000000  1.000000
8            emailservice  latency-50     25              2  2024-01-20T04:51:00Z    0.029478    0.024519  1.202252
9         checkoutservice         cpu     25              2  2024-01-20T04:43:00Z    0.548793    0.511585  1.072730
10        currencyservice  latency-90     25              2  2024-01-20T04:43:00Z    0.057071    0.053438  1.067986
11        checkoutservice  latency-50     25              2  2024-01-20T04:52:00Z    0.163646    0.154972  1.055972
12  recommendationservice         cpu     25              2  2024-01-20T04:49:00Z    3.054354    2.907853  1.050381
13           emailservice      socket     25              2  2024-01-20T04:54:00Z    9.783333    9.330000  1.048589
14  productcatalogservice         cpu     25              2  2024-01-20T04:41:00Z    4.402361    4.231176  1.040458
15        checkoutservice  latency-90     25              2  2024-01-20T04:52:00Z    0.251917    0.245148  1.027613
16  productcatalogservice    workload     25              2  2024-01-20T04:49:00Z  154.175400  150.339557  1.025515
17        currencyservice  latency-50     25              2  2024-01-20T04:43:00Z    0.004104    0.004009  1.023727
18  recommendationservice    workload     25              2  2024-01-20T04:41:00Z   20.369750   19.900880  1.023560
19           emailservice    workload     25              2  2024-01-20T04:57:00Z    1.241133    1.212977  1.023213```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse df_trace if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Parse timestamps (idempotent)
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Compute global P95 per (cmdb_id, trace_name) using the full trace.csv (before any filtering)
p95_global = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95_global')
)

# Incident window (UTC)
start = pd.to_datetime('2024-01-20T04:36:00Z', utc=True)
end = pd.to_datetime('2024-01-20T05:06:00Z', utc=True)

# Filter trace rows for the specified services and incident window
services = ["checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"]
df_trace_window = df_trace[
    df_trace['cmdb_id'].isin(services) &
    (df_trace['timestamp'] >= start) &
    (df_trace['timestamp'] <= end)
].copy()

# If no in-window rows, produce an empty summary with required columns
if df_trace_window.shape[0] == 0:
    summary_df_trace = pd.DataFrame(columns=[
        'cmdb_id','trace_name','count_in_window','anomaly_count','earliest_anom_ts',
        'max_value_in_window','p95_global','ratio'
    ])
    email_trace_rows = pd.DataFrame(columns=df_trace.columns)
else:
    # Merge global p95 into the windowed rows
    df_tw = df_trace_window.merge(p95_global, on=['cmdb_id','trace_name'], how='left')

    # Determine per-row anomaly: value >= p95_global (and p95_global not NaN)
    df_tw['is_anom'] = (df_tw['p95_global'].notna()) & (df_tw['value'] >= df_tw['p95_global'])

    # Aggregate within the window
    agg = df_tw.groupby(['cmdb_id','trace_name'], dropna=False).agg(
        count_in_window = ('value', 'count'),
        anomaly_count = ('is_anom', 'sum'),
        max_value_in_window = ('value', 'max'),
        p95_global = ('p95_global', 'first')
    ).reset_index()

    # earliest anomaly timestamp per group (if any)
    earliest = (
        df_tw[df_tw['is_anom']]
        .groupby(['cmdb_id','trace_name'], dropna=False)['timestamp']
        .min()
        .reset_index(name='earliest_anom_ts')
    )

    agg = agg.merge(earliest, on=['cmdb_id','trace_name'], how='left')

    # Format earliest_anom_ts as ISO UTC strings; leave as NA when no anomaly
    agg['earliest_anom_ts'] = pd.to_datetime(agg['earliest_anom_ts'], utc=True)
    agg['earliest_anom_ts'] = agg['earliest_anom_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    agg['earliest_anom_ts'] = agg['earliest_anom_ts'].where(agg['anomaly_count']>0, pd.NA)

    # Compute ratio = max_value_in_window / p95_global, handle p95==0 explicitly
    def compute_ratio(row):
        p = row['p95_global']
        m = row['max_value_in_window']
        if pd.isna(p):
            return np.nan
        if p == 0:
            return np.inf
        return m / p

    agg['ratio'] = agg.apply(compute_ratio, axis=1)

    # Round numeric columns for compactness
    agg['p95_global'] = pd.to_numeric(agg['p95_global'], errors='coerce').round(6)
    agg['max_value_in_window'] = pd.to_numeric(agg['max_value_in_window'], errors='coerce').round(6)
    agg['ratio'] = pd.to_numeric(agg['ratio'], errors='coerce')

    # Sort and limit to top 20 rows by anomaly_count desc then ratio desc
    summary_df_trace = agg[[
        'cmdb_id','trace_name','count_in_window','anomaly_count','earliest_anom_ts',
        'max_value_in_window','p95_global','ratio'
    ]].sort_values(['anomaly_count','ratio'], ascending=[False, False]).head(20).reset_index(drop=True)

    # Also return emailservice in-window trace rows where trace_name contains any of the target substrings
    targets = ['error_rate','duration_p95','duration_mean','row_count']
    pattern = '|'.join([t.replace('.', r'\.') for t in targets])  # safe pattern
    email_trace_rows = df_tw[
        (df_tw['cmdb_id'] == 'emailservice') &
        (df_tw['trace_name'].str.contains(pattern, regex=True))
    ].copy()

    # Format timestamp to ISO UTC strings and select relevant columns; limit to top 50 rows
    if email_trace_rows.shape[0] > 0:
        email_trace_rows['timestamp'] = pd.to_datetime(email_trace_rows['timestamp'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
        # keep a compact selection of columns
        email_trace_rows = email_trace_rows[['timestamp','cmdb_id','trace_name','value']].sort_values('timestamp').head(50).reset_index(drop=True)
    else:
        email_trace_rows = email_trace_rows.head(0)

# Display the two results (compact)
summary_df_trace, email_trace_rows

```
Out[3]:
```
```python
# Summarize trace anomaly findings in plain English using existing aggregated results:
# Uses summary_df_trace and email_trace_rows computed earlier.

from math import isfinite

# Prepare safety checks
if 'summary_df_trace' not in globals():
    summary = "No trace summary available."
else:
    df = summary_df_trace.copy()

    # Count groups with p95_global == 0 and anomaly_count > 0 (these produce inf ratio artifacts)
    zero_p95_groups = df[(df['p95_global'] == 0) & (df['anomaly_count'] > 0)]
    n_zero_p95 = zero_p95_groups.shape[0]

    # Non-trivial anomalies: p95_global > 0 and ratio > 1 and anomaly_count > 0
    nontrivial = df[(df['p95_global'] > 0) & (df['anomaly_count'] > 0) & (df['ratio'] > 1)].copy()
    nontrivial = nontrivial.sort_values(['ratio','anomaly_count'], ascending=[False, False]).head(5)

    # Format top non-trivial anomalies
    if nontrivial.shape[0] == 0:
        nontrivial_txt = "None (no trace groups with p95>0 and ratio>1 in the top results)."
    else:
        rows = []
        for _, r in nontrivial.iterrows():
            rows.append(
                f"{r['cmdb_id']} | {r['trace_name']} — anomalies={int(r['anomaly_count'])}, "
                f"earliest={r['earliest_anom_ts'] if pd.notna(r['earliest_anom_ts']) else 'NA'}, "
                f"ratio={float(r['ratio']):.3f}"
            )
        nontrivial_txt = "Top non-trivial anomalies: " + "; ".join(rows)

    # Summarize zero-p95 groups
    zero_p95_txt = (
        f"{n_zero_p95} trace groups (mostly error_rate traces) have global p95 == 0 and anomaly_count > 0; "
        "these produce infinite ratio and likely represent threshold/artifact rather than real degradations."
    ) if n_zero_p95 > 0 else "No groups with p95 == 0 producing infinite ratios."

    # Summarize emailservice in-window trace rows for the requested trace_name patterns
    if 'email_trace_rows' not in globals() or email_trace_rows.shape[0] == 0:
        email_txt = "No in-window trace rows for emailservice matching the requested trace_name patterns."
    else:
        et = email_trace_rows.copy()
        # Aggregate simple stats per trace_name
        stats = et.groupby('trace_name')['value'].agg(['count','min','median','max']).reset_index()
        # Format into short sentences
        parts = []
        for _, r in stats.iterrows():
            parts.append(
                f"{r['trace_name']}: n={int(r['count'])}, min={r['min']:.6g}, median={r['median']:.6g}, max={r['max']:.6g}"
            )
        email_txt = (
            f"emailservice in-window trace rows (target traces) count={et.shape[0]}. "
            "Per-trace stats: " + "; ".join(parts)
        )

    # Compose final summary
    summary = (
        "Trace analysis summary (incident window 2024-01-20T04:36:00Z to 2024-01-20T05:06:00Z):\n"
        f"- Note: Many top anomalous trace groups are 'error_rate' series with global p95 == 0, causing infinite ratios.\n"
        f"- Zero-p95 groups: {zero_p95_txt}\n"
        f"- Non-trivial anomalies (p95>0 and ratio>1): {nontrivial_txt}\n"
        f"- emailservice targeted traces: {email_txt}\n\n"
        "Interpretation: Most flagged anomalies are artifacts of p95==0 for error_rate traces. "
        "There are few meaningful trace-level exceedances (p95>0 & ratio>1); those listed above deserve follow-up if you want specific trace-level investigation."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                   trace_name  count_in_window  anomaly_count      earliest_anom_ts  max_value_in_window  p95_global  ratio
0         currencyservice        trace.from_frontendservice.error_rate               25             25  2024-01-20T04:39:00Z                  0.0         0.0    inf
1         checkoutservice        trace.from_checkoutservice.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
2         checkoutservice        trace.from_frontendservice.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
3         checkoutservice                   trace.from_root.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
4         checkoutservice          trace.to_checkoutservice.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
5         checkoutservice          trace.to_currencyservice.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
6         checkoutservice             trace.to_emailservice.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
7         checkoutservice           trace.to_paymentservice.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
8         checkoutservice    trace.to_productcatalogservice.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
9         currencyservice        trace.from_checkoutservice.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
10        currencyservice                   trace.from_root.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
11           emailservice        trace.from_checkoutservice.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
12           emailservice                   trace.from_root.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
13  productcatalogservice        trace.from_checkoutservice.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
14  productcatalogservice        trace.from_frontendservice.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
15  productcatalogservice  trace.from_recommendationservice.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
16  productcatalogservice                   trace.from_root.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
17  recommendationservice        trace.from_frontendservice.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
18  recommendationservice  trace.from_recommendationservice.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf
19  recommendationservice                   trace.from_root.error_rate               24             24  2024-01-20T04:39:00Z                  0.0         0.0    inf,                timestamp       cmdb_id                                trace_name      value
0   2024-01-20T04:39:00Z  emailservice  trace.from_checkoutservice.duration_mean   0.000317
1   2024-01-20T04:39:00Z  emailservice   trace.from_checkoutservice.duration_p95   0.000475
2   2024-01-20T04:39:00Z  emailservice     trace.from_checkoutservice.error_rate   0.000000
3   2024-01-20T04:39:00Z  emailservice      trace.from_checkoutservice.row_count  26.000000
4   2024-01-20T04:39:00Z  emailservice             trace.from_root.duration_mean   0.000040
..                   ...           ...                                       ...        ...
45  2024-01-20T04:44:00Z  emailservice  trace.from_checkoutservice.duration_mean   0.000312
46  2024-01-20T04:44:00Z  emailservice      trace.from_checkoutservice.row_count  32.000000
47  2024-01-20T04:44:00Z  emailservice   trace.from_checkoutservice.duration_p95   0.000529
48  2024-01-20T04:45:00Z  emailservice                 trace.from_root.row_count  24.000000
49  2024-01-20T04:45:00Z  emailservice  trace.from_checkoutservice.duration_mean   0.000355

[50 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse df_metric if available; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Parse timestamps (idempotent)
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full metric series
p95_metric = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95_global')
)

# 2) Incident window (UTC) and filter to emailservice within window
start = pd.to_datetime('2024-01-20T04:36:00Z', utc=True)
end = pd.to_datetime('2024-01-20T05:06:00Z', utc=True)

df_email_win = (
    df_metric[
        (df_metric['cmdb_id'] == 'emailservice') &
        (df_metric['timestamp'] >= start) &
        (df_metric['timestamp'] <= end)
    ]
    .merge(p95_metric[p95_metric['cmdb_id'] == 'emailservice'][['kpi_name','p95_global']],
           on='kpi_name', how='left')
    .sort_values(['kpi_name','timestamp'])
    .reset_index(drop=True)
)

# 3) For each kpi_name, identify consecutive sequences where value >= p95_global
sequences = []
if df_email_win.shape[0] > 0:
    for kpi, grp in df_email_win.groupby('kpi_name', sort=True):
        g = grp.copy()
        # mark anomaly rows where value >= global p95 (and p95 not NaN)
        g['is_anom'] = (g['p95_global'].notna()) & (g['value'] >= g['p95_global'])
        if g['is_anom'].sum() == 0:
            continue
        # Keep only anomaly rows and detect consecutive 1-minute spacing
        anom = g[g['is_anom']].copy().reset_index(drop=True)
        # compute seconds diff between successive anomaly timestamps
        anom['diff_s'] = anom['timestamp'].diff().dt.total_seconds().fillna(np.inf)
        # start new group when diff != 60
        anom['seq_id'] = (anom['diff_s'] != 60).cumsum()
        # aggregate sequences
        seqs = (
            anom.groupby('seq_id', sort=True)
            .agg(
                kpi_name = ('kpi_name', 'first'),
                start_ts = ('timestamp', 'min'),
                end_ts = ('timestamp', 'max'),
                length_minutes = ('timestamp', 'count'),  # rows are per-minute -> count = minutes
                max_value_in_sequence = ('value', 'max'),
                p95_global = ('p95_global', 'first')
            )
            .reset_index(drop=True)
        )
        # keep sequences length >= 2 minutes
        seqs = seqs[seqs['length_minutes'] >= 2].copy()
        if seqs.shape[0] == 0:
            continue
        # compute ratio, handle p95==0 or NaN
        def ratio_val(row):
            p = row['p95_global']
            m = row['max_value_in_sequence']
            if pd.isna(p):
                return np.nan
            if p == 0:
                return np.inf
            return m / p
        seqs['ratio'] = seqs.apply(ratio_val, axis=1)
        # append
        sequences.append(seqs)

# Concatenate sequences
if len(sequences) == 0:
    seq_df = pd.DataFrame(columns=[
        'kpi_name','start_ts','end_ts','length_minutes','max_value_in_sequence','p95_global','ratio'
    ])
    earliest_fault_start = 'none'
else:
    seq_df = pd.concat(sequences, ignore_index=True)
    # format timestamps as ISO UTC strings
    seq_df['start_ts'] = pd.to_datetime(seq_df['start_ts'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    seq_df['end_ts'] = pd.to_datetime(seq_df['end_ts'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    # round numeric columns for compactness
    seq_df['max_value_in_sequence'] = pd.to_numeric(seq_df['max_value_in_sequence'], errors='coerce').round(6)
    seq_df['p95_global'] = pd.to_numeric(seq_df['p95_global'], errors='coerce').round(6)
    # Make ratio display-friendly: keep numeric inf as np.inf (will show as inf)
    seq_df['ratio'] = pd.to_numeric(seq_df['ratio'], errors='coerce')
    # select and order columns as requested
    seq_df = seq_df[[
        'kpi_name','start_ts','end_ts','length_minutes','max_value_in_sequence','p95_global','ratio'
    ]]
    # sort by ratio desc, limit to top 20
    seq_df = seq_df.sort_values('ratio', ascending=False).head(20).reset_index(drop=True)
    # earliest fault start_ts among these sequences
    try:
        earliest_fault_start = seq_df['start_ts'].min()
        if pd.isna(earliest_fault_start):
            earliest_fault_start = 'none'
    except Exception:
        earliest_fault_start = 'none'

# Display results (compact)
seq_df, earliest_fault_start

```
Out[5]:
```
```python
# Summarize emailservice metric-level fault identification as a single string.
# Reuse seq_df and earliest_fault_start from the previous computation.

try:
    seq_df
except NameError:
    seq_df = None

if seq_df is None or seq_df.shape[0] == 0:
    summary = "No qualifying metric sequences (>=2 minutes) found for emailservice in the incident window."
else:
    total_seqs = seq_df.shape[0]
    # Identify diskio (p95==0) artifact rows
    diskio_rows = seq_df[(seq_df['kpi_name']=='diskio') & (seq_df['p95_global']==0)]
    n_diskio = diskio_rows.shape[0]
    # Meaningful sequences where p95_global > 0
    meaningful = seq_df[seq_df['p95_global'] > 0].copy().sort_values('ratio', ascending=False)
    # Prepare text for top meaningful sequences (up to 5)
    meaningful_parts = []
    for _, r in meaningful.head(5).iterrows():
        meaningful_parts.append(
            f"{r['kpi_name']}: {r['start_ts']} to {r['end_ts']} ({int(r['length_minutes'])} min), "
            f"max={r['max_value_in_sequence']}, p95={r['p95_global']}, ratio={float(r['ratio']):.3f}"
        )
    if len(meaningful_parts) == 0:
        meaningful_txt = "No meaningful sequences with p95>0 found."
    else:
        meaningful_txt = " ; ".join(meaningful_parts)
    # Earliest fault start among sequences (as computed)
    earliest = earliest_fault_start if 'earliest_fault_start' in globals() else 'none'
    # Also earliest meaningful start if exists
    if meaningful.shape[0] > 0:
        earliest_meaningful = meaningful['start_ts'].min()
    else:
        earliest_meaningful = 'none'

    summary = (
        f"Emailservice metric fault summary (window 2024-01-20T04:36:00Z to 2024-01-20T05:06:00Z):\n"
        f"- Total qualifying sequences (length>=2 min): {total_seqs}.\n"
        f"- Diskio sequences with p95==0 (artifact-like): {n_diskio} (e.g. diskio 2024-01-20T04:39:00Z to 2024-01-20T05:03:00Z; ratio=inf) — treat with caution.\n"
        f"- Meaningful exceedances (p95>0), top by ratio: {meaningful_txt}\n"
        f"- Earliest sequence start among these: {earliest}\n"
        f"- Earliest meaningful (p95>0) sequence start: {earliest_meaningful}\n\n"
        "Interpretation: The strongest meaningful metric signal is 'latency-50' (04:51–04:52) with ratio ≈1.20 over global p95. "
        "Socket shows a modest exceedance (~1.05). Diskio entries are likely artifacts because global p95==0."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     kpi_name              start_ts                end_ts  length_minutes  max_value_in_sequence  p95_global     ratio
0      diskio  2024-01-20T04:39:00Z  2024-01-20T05:03:00Z              25               0.000000    0.000000       inf
1  latency-50  2024-01-20T04:51:00Z  2024-01-20T04:52:00Z               2               0.029478    0.024519  1.202252
2      socket  2024-01-20T04:54:00Z  2024-01-20T04:55:00Z               2               9.783333    9.330000  1.048589, '2024-01-20T04:39:00Z')```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse existing df_log and df_error if present, otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')
try:
    df_error
except NameError:
    df_error = pd.read_csv('error_logs.csv')

# Parse timestamps (idempotent)
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start = pd.to_datetime('2024-01-20T04:36:00Z', utc=True)
end = pd.to_datetime('2024-01-20T05:06:00Z', utc=True)

# 1) Compute global P95 per (cmdb_id, log_name) using full log.csv BEFORE filtering
p95_log_global = (
    df_log
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95_global')
)

# Filter log.csv to emailservice within incident window
email_logs_win = df_log[
    (df_log['cmdb_id'] == 'emailservice') &
    (df_log['timestamp'] >= start) &
    (df_log['timestamp'] <= end)
].copy().sort_values('timestamp').reset_index(drop=True)

# Group summary by log_name for emailservice (count, min_ts, max_ts), then join p95_global
if email_logs_win.shape[0] == 0:
    email_log_group = pd.DataFrame(columns=['log_name','count','min_ts','max_ts','p95'])
else:
    grp = (
        email_logs_win
        .groupby('log_name', dropna=False)['timestamp']
        .agg(count='count', min_ts='min', max_ts='max')
        .reset_index()
    )
    # merge p95 (global) for emailservice log_name
    p95_email = p95_log_global[p95_log_global['cmdb_id'] == 'emailservice'][['log_name','p95_global']]
    grp = grp.merge(p95_email, on='log_name', how='left')
    # format timestamps as ISO UTC strings and round p95
    grp['min_ts'] = pd.to_datetime(grp['min_ts'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    grp['max_ts'] = pd.to_datetime(grp['max_ts'], utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    grp['p95'] = pd.to_numeric(grp['p95_global'], errors='coerce').round(6)
    email_log_group = grp[['log_name','count','min_ts','max_ts','p95']]

# Prepare up to 20 raw filtered log rows with timestamp ISO, log_name, value
if email_logs_win.shape[0] == 0:
    email_log_rows = email_logs_win.head(0)[['timestamp','log_name','value']]
else:
    rows = email_logs_win.copy()
    rows['timestamp'] = rows['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    email_log_rows = rows[['timestamp','log_name','value']].head(20).reset_index(drop=True)

# 2) Filter error_logs.csv for emailservice within incident window and return up to 20 rows (timestamp ISO, message)
email_errors_win = df_error[
    (df_error['cmdb_id'] == 'emailservice') &
    (df_error['timestamp'] >= start) &
    (df_error['timestamp'] <= end)
].copy().sort_values('timestamp').reset_index(drop=True)

if email_errors_win.shape[0] == 0:
    # Explicit empty result DataFrame with the expected columns
    email_error_rows = pd.DataFrame(columns=['timestamp','message'])
else:
    er = email_errors_win.copy()
    er['timestamp'] = er['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    email_error_rows = er[['timestamp','message']].head(20).reset_index(drop=True)

# Display compact results: grouped summary for emailservice logs, up to 20 raw email log rows, up to 20 raw email error rows
email_log_group, email_log_rows, email_error_rows

```
Out[6]:
```
Summary (plain English):

- Grouped log summary for emailservice (incident window 2024-01-20T04:36:00Z–2024-01-20T05:06:00Z):
  - log.error_count — count=24, window 2024-01-20T04:39:00Z to 2024-01-20T05:02:00Z, global P95=0.0.
  - log.row_count   — count=24, window 2024-01-20T04:39:00Z to 2024-01-20T05:02:00Z, global P95=32.0.

- Raw filtered log observations (up to 20 rows shown):
  - log.error_count is 0 at every sampled minute in the window (no error spikes).
  - log.row_count fluctuates (examples): 26@04:39, 28@04:40, 23@04:41, 28@04:42, 32@04:43, 32@04:44, 21@04:45, 26@04:46, 24@04:47, 30@04:48. Peak values (32) occur (e.g., 04:43–04:44), matching the global P95.

- error_logs for emailservice in the incident window:
  - No error log rows found (empty result).

Interpretation: emailservice shows no error log entries and log.error_count remains zero. log.row_count occasionally reaches the global P95 (32) at a few minutes but otherwise varies below it. There is no evidence of error spikes from logs in this window.

The original code execution output of IPython Kernel is also provided below for reference:

(          log_name  count                min_ts                max_ts   p95
0  log.error_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z   0.0
1    log.row_count     24  2024-01-20T04:39:00Z  2024-01-20T05:02:00Z  32.0,                timestamp         log_name  value
0   2024-01-20T04:39:00Z  log.error_count      0
1   2024-01-20T04:39:00Z    log.row_count     26
2   2024-01-20T04:40:00Z  log.error_count      0
3   2024-01-20T04:40:00Z    log.row_count     28
4   2024-01-20T04:41:00Z  log.error_count      0
5   2024-01-20T04:41:00Z    log.row_count     23
6   2024-01-20T04:42:00Z  log.error_count      0
7   2024-01-20T04:42:00Z    log.row_count     28
8   2024-01-20T04:43:00Z  log.error_count      0
9   2024-01-20T04:43:00Z    log.row_count     32
10  2024-01-20T04:44:00Z    log.row_count     32
11  2024-01-20T04:44:00Z  log.error_count      0
12  2024-01-20T04:45:00Z  log.error_count      0
13  2024-01-20T04:45:00Z    log.row_count     21
14  2024-01-20T04:46:00Z  log.error_count      0
15  2024-01-20T04:46:00Z    log.row_count     26
16  2024-01-20T04:47:00Z  log.error_count      0
17  2024-01-20T04:47:00Z    log.row_count     24
18  2024-01-20T04:48:00Z  log.error_count      0
19  2024-01-20T04:48:00Z    log.row_count     30, Empty DataFrame
Columns: [timestamp, message]
Index: [])```
```